# Classical Machine Learning Models

## Objective

This notebook trains and evaluates classical machine learning models using the handcrafted image features extracted in the previous notebook.

The primary objectives are:

- Load the engineered feature dataset
- Prepare the data for model training
- Train Random Forest and XGBoost classifiers
- Evaluate both models using Stratified 5-Fold Cross Validation
- Compare their performance
- Analyze feature importance
- Save the trained XGBoost model for future inference

## Load the Engineered Feature Dataset

Load the handcrafted feature dataset generated during feature engineering. This dataset serves as the input for all classical machine learning experiments.

In [ ]:
import pandas as pd

df = pd.read_csv("../engineered_features.csv")

In [ ]:
df.to_csv("../engineered_features.csv", index=False)

## Encode Class Labels

Convert the categorical target labels into numerical values suitable for machine learning algorithms.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["label"] = le.fit_transform(df["label"])

df.head()

## Separate Features and Target

Split the dataset into input features and the target class that will be predicted by the classifiers.

In [ ]:
X = df.drop(columns=[
    "label",
    "filename",
    "width",
    "height"
])

y = df["label"]

## Train-Test Split

Create a training and testing split that can be used for initial experimentation before cross-validation.

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

## Train the Random Forest Classifier

Initialize and train a Random Forest classifier using the engineered image features.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(

    n_estimators=300,

    random_state=42

)

rf.fit(X_train,y_train)

## Evaluate Random Forest Performance

Measure the predictive performance of the trained Random Forest model using standard classification metrics.

In [ ]:
from sklearn.metrics import accuracy_score

pred = rf.predict(X_test)

print(accuracy_score(y_test,pred))

## Analyze Feature Importance

Visualize the importance assigned to each handcrafted feature by the Random Forest model.

This provides insight into which image characteristics contribute most to distinguishing real photographs from screen recaptures.

In [ ]:
importance = pd.DataFrame({

    "Feature":X.columns,

    "Importance":rf.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

importance.head(20)

## Import Cross Validation Utilities

Import Stratified K-Fold Cross Validation to obtain reliable model evaluation across multiple dataset splits.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier

## Configure Random Forest

Define the Random Forest model with the selected hyperparameters.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

## Configure Stratified 5-Fold Cross Validation

Create a Stratified K-Fold splitter to preserve class balance across every validation fold.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## Define Evaluation Metrics

Specify the evaluation metrics that will be computed during cross-validation.

The selected metrics include:

- Accuracy
- Precision
- Recall
- F1 Score

In [ ]:
scoring = [
    "accuracy",
    "precision",
    "recall",
    "f1"
]

scores = cross_validate(
    rf,
    X,
    y,
    cv=cv,
    scoring=scoring
)

## Cross Validation Results

Display the evaluation scores obtained on every validation fold.

In [ ]:
print("Accuracy :", scores["test_accuracy"])
print()

print("Precision:", scores["test_precision"])
print()

print("Recall   :", scores["test_recall"])
print()

print("F1 Score :", scores["test_f1"])

## Average Random Forest Performance

Compute the mean performance across all folds to estimate the generalization ability of the Random Forest classifier.

In [ ]:
print(f"Mean Accuracy : {scores['test_accuracy'].mean():.4f}")
print(f"Std Accuracy  : {scores['test_accuracy'].std():.4f}")

print()

print(f"Mean Precision : {scores['test_precision'].mean():.4f}")
print(f"Mean Recall    : {scores['test_recall'].mean():.4f}")
print(f"Mean F1 Score  : {scores['test_f1'].mean():.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(
    range(1,6),
    scores["test_accuracy"],
    marker="o",
    linewidth=2
)

plt.xticks(range(1,6))
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.title("5-Fold Cross Validation Accuracy")

plt.grid(True)

plt.show()

## Import XGBoost

Import the XGBoost classifier for the second classical machine learning experiment.

In [ ]:
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate

## Configure the XGBoost Classifier

Initialize the XGBoost model using the selected training parameters.

In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## Evaluate XGBoost

Run Stratified 5-Fold Cross Validation and collect the evaluation scores for every fold.

In [ ]:
scoring = [
    "accuracy",
    "precision",
    "recall",
    "f1"
]

scores = cross_validate(
    xgb,
    X,
    y,
    cv=cv,
    scoring=scoring
)

In [ ]:
print("Accuracy :", scores["test_accuracy"])
print()

print("Precision:", scores["test_precision"])
print()

print("Recall   :", scores["test_recall"])
print()

print("F1 Score :", scores["test_f1"])

In [ ]:
print(f"Mean Accuracy : {scores['test_accuracy'].mean():.4f}")
print(f"Std Accuracy  : {scores['test_accuracy'].std():.4f}")

print()

print(f"Mean Precision : {scores['test_precision'].mean():.4f}")
print(f"Mean Recall    : {scores['test_recall'].mean():.4f}")
print(f"Mean F1 Score  : {scores['test_f1'].mean():.4f}")

## Train the Final XGBoost Model

Train the XGBoost classifier using the complete engineered feature dataset before deployment.

In [ ]:
xgb.fit(X, y)

## Prepare Feature Importance

Create a DataFrame containing the learned feature importance scores.

In [ ]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(20)

In [ ]:
import matplotlib.pyplot as plt

top = importance.head(20)

plt.figure(figsize=(10,6))

plt.barh(top["Feature"], top["Importance"])

plt.gca().invert_yaxis()

plt.title("Top 20 XGBoost Feature Importance")

plt.show()

## Save the Trained Model

Store the trained XGBoost model so that it can be reused without retraining.

In [ ]:
xgb.save_model("../models/xgb_model.json")

# Summary

This notebook established classical machine learning baselines for the screen recapture detection task using handcrafted image features.

### Models Evaluated

- Random Forest
- XGBoost

### Evaluation Strategy

- Stratified 5-Fold Cross Validation
- Accuracy
- Precision
- Recall
- F1 Score

### Results

The Random Forest classifier achieved a mean accuracy of approximately **80.29%**, while XGBoost improved the overall performance to approximately **83.29%**.

Feature importance analysis showed that frequency-domain and texture-based descriptors contributed most significantly to the classification task.

Although these classical models produced competitive results, the subsequent notebooks investigate transfer learning with convolutional neural networks to further improve classification performance.